# Phase 2.2 : Intégration Dropout + réentraînement

Chats vs chiens (Microsoft Cats vs Dogs Dataset). Suite de `phase2_1_augmentation.ipynb`.

**Ce notebook est autonome** : la section "Reprise" ci-dessous refait le setup des phases 1.1, 1.2 et 2.1 (téléchargement, organisation, pipeline de données, définition de la data augmentation), sans les explications déjà vues.

## Reprise (setup phases 1.1, 1.2 et 2.1, condensé)

Nécessite un `kaggle.json` (voir phase 1.1 pour l'obtenir).

In [ ]:
CLASS_A = "cat"
CLASS_B = "dog"
DATA_ROOT = "/content/data"

In [ ]:
!pip install kaggle -q
!mkdir -p ~/.kaggle

In [ ]:
from google.colab import files
files.upload()  # sélectionner kaggle.json

import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

In [ ]:
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p /content/data/
!cd /content/data && unzip -q microsoft-catsvsdogs-dataset.zip

In [ ]:
import os, shutil, random
import tensorflow as tf

RAW_DIR = os.path.join(DATA_ROOT, "PetImages")
raw_dirs = {CLASS_A: os.path.join(RAW_DIR, "Cat"), CLASS_B: os.path.join(RAW_DIR, "Dog")}


def list_valid_images(folder):
    # Ni PIL.Image.verify() ni la verification d'entete JFIF ne suffisent : ce dataset
    # contient quelques fichiers que seul le decodeur TensorFlow rejette au moment de
    # l'entrainement (InvalidArgumentError "Unknown image file format"). On teste donc
    # directement avec tf.io.decode_image, le meme decodeur utilise par
    # image_dataset_from_directory en phase 1.2.
    valid = []
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        if os.path.getsize(path) == 0:
            continue
        try:
            img_bytes = tf.io.read_file(path)
            tf.io.decode_image(img_bytes, channels=3)
        except Exception:
            continue
        valid.append(path)
    return valid


files_a = list_valid_images(raw_dirs[CLASS_A])
files_b = list_valid_images(raw_dirs[CLASS_B])

for split in ["train", "val"]:
    for cls in [CLASS_A, CLASS_B]:
        os.makedirs(os.path.join(DATA_ROOT, split, cls), exist_ok=True)

random.seed(42)
random.shuffle(files_a)
random.shuffle(files_b)


def split_and_copy(file_list, cls):
    cut = int(len(file_list) * 0.8)
    train_files, val_files = file_list[:cut], file_list[cut:]
    for f in train_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "train", cls, os.path.basename(f)))
    for f in val_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "val", cls, os.path.basename(f)))


split_and_copy(files_a, CLASS_A)
split_and_copy(files_b, CLASS_B)
print(f"{CLASS_A}: {len(files_a)} images valides")
print(f"{CLASS_B}: {len(files_b)} images valides")

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "train"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
)
# Filet de securite : le decodeur d'images en mode graphe (utilise ici) est parfois plus
# strict que tf.io.decode_image en mode eager (utilise a l'organisation des dossiers).
# ignore_errors() saute silencieusement tout exemple qui ferait planter le decodeur.
train_ds = train_ds.apply(tf.data.experimental.ignore_errors())

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "val"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42,
)
val_ds = val_ds.apply(tf.data.experimental.ignore_errors())

normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# Ces couches sont actives uniquement en mode training (model.fit) et
# passives en mode inference (model.predict, model.evaluate).
# Consequence : la val_accuracy est calculee sur les images ORIGINALES. C'est voulu.
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),   # +/-10% de 360 deg = +/-36 deg
    layers.RandomZoom(0.1),       # +/-10% de zoom
], name="data_augmentation")

In [ ]:
import matplotlib.pyplot as plt


def plot_history(history, title=""):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history["loss"], label="Train loss")
    ax1.plot(history.history["val_loss"], label="Val loss")
    ax1.set_title(f"{title} - Loss")
    ax1.legend()

    ax2.plot(history.history["accuracy"], label="Train accuracy")
    ax2.plot(history.history["val_accuracy"], label="Val accuracy")
    ax2.set_title(f"{title} - Accuracy")
    ax2.legend()

    plt.tight_layout()
    plt.savefig(f"curves_{title.lower().replace(' ', '_').replace('+', '')}.png", dpi=100)
    plt.show()

## Phase 2.2 : Intégration Dropout + réentraînement

**Objectif** : construire un CNN augmenté avec Dropout, l'entraîner depuis zéro sur le même dataset que la phase 1.4, et comparer les courbes directement avec celles du CNN from scratch.

In [ ]:
from tensorflow.keras import layers, models


def build_cnn_augmented(input_shape):
    """CNN avec data augmentation integree + Dropout.
    Meme architecture de base que build_cnn_scratch (phase 1.3), mais avec regularisation.
    """
    model = models.Sequential([
        data_augmentation,

        layers.Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        # Dropout APRES Flatten (pas entre les blocs Conv) : les feature maps spatiales
        # ne doivent pas etre coupees, seulement les activations apres aplatissement.
        layers.Dropout(0.4),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    return model


model_augmented = build_cnn_augmented(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
model_augmented.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model_augmented.summary()

In [ ]:
import datetime, time

log_dir_aug = "logs/augmented/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback_aug = tf.keras.callbacks.TensorBoard(log_dir=log_dir_aug, histogram_freq=1)

early_stopping_aug = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

start = time.time()

history_augmented = model_augmented.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    callbacks=[tensorboard_callback_aug, early_stopping_aug],
)

training_time_augmented = time.time() - start
print(f"Temps d'entrainement : {training_time_augmented:.0f}s")
print(f"val_accuracy finale : {max(history_augmented.history['val_accuracy']):.3f}")

In [ ]:
plot_history(history_augmented, "CNN augmente + Dropout")

In [ ]:
import json as _json

with open("history_augmented.json", "w") as f:
    _json.dump(history_augmented.history, f)
model_augmented.save("model_augmented.keras")
print("history_augmented.json et model_augmented.keras sauvegardes.")

### Ce que tu dois observer

La `val_accuracy` monte plus haut qu'en phase 1.4 (typiquement 80-85% sur ce dataset). La divergence train/val est réduite. La convergence est peut-être plus lente — le modèle ne peut plus mémoriser aussi facilement grâce à l'augmentation + Dropout. C'est le signe que la régularisation fonctionne.

### Vérifications avant la phase 2.3

- **Happy path** : `val_accuracy` supérieure à la phase 1.4, gap train/val réduit (overfit atténué mais pas nul), `EarlyStopping` se déclenche plus tard qu'en phase 1.4.
- **Edge case** : déplacer `Dropout` avant `Flatten` couperait des feature maps spatiales entières (positions, pas juste des activations) — la `val_accuracy` chuterait par rapport à la version post-Flatten.
- **Adversarial** : passer à `Dropout(0.8)` ferait s'effondrer la `train_accuracy` sous 70% — trop de signal coupé à chaque forward, le modèle n'apprend plus. Le bon range pour un CNN est 0.2-0.4 après Flatten.

**Prochaine étape** : Phase 2.3 — comparaison chiffrée entre le CNN from scratch (phase 1.4) et ce CNN augmenté.